# GVK Private AI — QLoRA Fine-Tuning & GGUF Adapter Exporter
**Базова модель:** `Qwen/Qwen2.5-7B-Instruct` (`unsloth/Qwen2.5-7B-Instruct-bnb-4bit`)  
**Призначення:** Навчання 3-х індивідуальних LoRA-адаптерів (`friend_1`, `friend_2`, `friend_3`) та їх експорт у формат `.gguf` для миттєвого перемикання в `llama-server`.

--- 
### Перед початком роботи:
1. У верхньому меню Google Colab натисніть **Runtime -> Change runtime type**.
2. Оберіть **Hardware accelerator: T4 GPU** (або L4 / A100) та натисніть **Save**.
3. Запускайте комірки послідовно зверху вниз (`Shift + Enter`).

## Крок 1. Перевірка GPU та встановлення залежностей (Unsloth + llama.cpp)

In [ ]:
%%capture
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

# Клонуємо llama.cpp для офіційного скрипта конвертації LoRA -> GGUF
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!pip install -r /content/llama.cpp/requirements/requirements-convert_lora_to_gguf.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU не знайдено! Увімкніть T4 GPU у меню Runtime -> Change runtime type."
print(f"✅ GPU підключено: {torch.cuda.get_device_name(0)}")

## Крок 2. Завантаження датасету `.jsonl` у тимчасову пам'ять Colab
Оберіть, якого саме агента навчаємо зараз (`friend_1`, `friend_2` або `friend_3`), і завантажте відповідний файл `*_chatml.jsonl`.

In [ ]:
import os
import json
from google.colab import files

# Вкажіть ID агента, якого тренуємо в цьому запуску:
PERSONA_ID = "friend_1"  # Змініть на "friend_2" або "friend_3" для наступних друзів

os.makedirs("/content/datasets", exist_ok=True)
print(f"📂 Завантажте файл {PERSONA_ID}_chatml.jsonl через вікно вибору файлу нижче:")
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
DATASET_PATH = f"/content/datasets/{uploaded_filename}"
os.replace(uploaded_filename, DATASET_PATH)

# Валідація завантаженого файлу
raw_samples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if line.strip():
            obj = json.loads(line)
            assert "messages" in obj and obj["messages"][-1]["role"] == "assistant", f"Помилка в рядку {i}"
            raw_samples.append(obj)

print(f"✅ Датасет '{DATASET_PATH}' успішно перевірено! Кількість діалогів: {len(raw_samples)}")

## Крок 3. Ініціалізація базової моделі `Qwen2.5-7B-Instruct` та матриць QLoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

MAX_SEQ_LENGTH = 2048
BASE_MODEL_UNSLOTH = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
BASE_MODEL_HF = "Qwen/Qwen2.5-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_UNSLOTH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("✅ Базова модель та LoRA-шари (r=32, alpha=64) готові до навчання!")

## Крок 4. Маскування лоссу (`train_on_responses_only`) та старт навчання
Модель навчається **виключно на відповідях цільового друга** (`<|im_start|>assistant`), ігноруючи помилки на репліках інших учасників.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

def format_chatml(messages):
    out = []
    for m in messages:
        out.append(f"<|im_start|>{m['role']}\n{m['content'].strip()}<|im_end|>\n")
    return "".join(out)

formatted_texts = [format_chatml(item["messages"]) for item in raw_samples]
hf_dataset = Dataset.from_dict({"text": formatted_texts})

training_args = SFTConfig(
    output_dir=f"/content/checkpoints_{PERSONA_ID}",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=1.8,
    learning_rate=1.5e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    args=training_args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print(f"🚀 Починаємо навчання QLoRA для [{PERSONA_ID}]...")
stats = trainer.train()
print(f"🎉 Навчання завершено! Фінальний Loss: {stats.metrics.get('train_loss', 0.0):.4f}")

## Крок 5. Конвертація у формат `.gguf` та завантаження на комп'ютер
Ця комірка зберігає адаптер, конвертує його у файл `friend_X_lora.gguf` (F16) і автоматично скачує вам у браузер.

In [ ]:
import subprocess
from google.colab import files

SAFETENSORS_DIR = f"/content/hf_lora_{PERSONA_ID}"
GGUF_FILENAME = f"{PERSONA_ID}_lora.gguf"
GGUF_PATH = f"/content/{GGUF_FILENAME}"

model.save_pretrained(SAFETENSORS_DIR)
tokenizer.save_pretrained(SAFETENSORS_DIR)

print("⚙️ Конвертація Safetensors LoRA -> GGUF (F16)...")
cmd = [
    "python3",
    "/content/llama.cpp/convert_lora_to_gguf.py",
    "--base", BASE_MODEL_HF,
    "--outtype", "f16",
    "--outfile", GGUF_PATH,
    SAFETENSORS_DIR
]
subprocess.run(cmd, check=True)

size_mb = os.path.getsize(GGUF_PATH) / (1024 * 1024)
print(f"✅ Готово! Файл {GGUF_FILENAME} ({size_mb:.2f} МБ) створено. Починається завантаження у браузер...")
files.download(GGUF_PATH)